In [ ]:
# 3D Condor debug
electron_density = np.abs(fft.ifftn(fft.fftshift(res3d['entry_1']['data_1']['data_fourier'])))
plt.imshow(fft.fftshift(electron_density)[108]);
plt.colorbar();

#np.save('intensity.npy', intensity3d)
#np.save('electron_density.npy', electron_density)

import scipy.fft as fft

auto_corr = np.abs(fft.ifftn(fft.fftshift(intensity3d)))

plt.figure(dpi=150)
plt.imshow(fft.fftshift(auto_corr)[108]**0.7, vmin=0, vmax=0.03, interpolation=None)
plt.colorbar();

In [ ]:
# Using the auto-correlation as a way to define initial support to shrink
auto_corr = np.abs(fft.ifftn(fft.fftshift(frame)))
support_autocorr = fft.fftshift(auto_corr > 0.001)
support_autocorr.sum()

In [ ]:
plt.figure(dpi=150)
plt.imshow(support_autocorr[182], cmap='gray')
plt.xticks([])
plt.yticks([])
plt.colorbar();

In [ ]:
#SBATCH --constraint='EPYC&'

In [ ]:
# 3D support thresholding
eleactron_density = np.abs(fft.ifftn(fft.fftshift(res3d['entry_1']['data_1']['data_fourier'])))
support = (electron_density > 0.0004)
plt.imshow(fft.fftshift(support)[:,:,108]);
plt.colorbar();

#np.save('static_support.npy', support)

electron_density[electron_density > 0.].max()

In [ ]:
# Linear combinations
with mrcfile.open('emc_fit.mrc', mode='r') as f_mod:
    intens = f_mod.data

plt.imshow(intens[:,:,108], vmin=0, vmax=0.03, interpolation=None)
plt.colorbar();

#np.save('emc_intens_fit.npy', intens)

In [ ]:
with h5py.File('gt_intens.h5', mode='r') as test:
    gt_intens = test['gt'][:]

plt.imshow(gt_intens[:,:,108], vmin=0, vmax=0.3, interpolation=None)
plt.colorbar();

# Linear combination of EMC + Condor models

#a_emc = 1.0 #1
#b_gt = 0.1  #1

#a_emc = 0.5 #2
#b_gt = 0.5 #2

#a_emc = 10.0 #3
#b_gt = 1.0 #3

a_emc = 10.0 #4
b_gt = 0.1 #4

lin_comb = a_emc * intens + b_gt * gt_intens

plt.imshow(lin_comb[:,:,108], vmin=0, vmax=2.0, interpolation=None)
plt.colorbar();

#np.save('lin_comb_fit_4.npy', lin_comb)

In [ ]:
alg = "raar"

n_recons = 2
niter_alg = 500
niter_er = 50
niter_store = 5  # number of output images

beta_start = 0.99
beta_end = 0.99

i_frac, f_frac = 1.1, 1.0
volume_i, volume_f = i_frac * vol_frac, f_frac * vol_frac

blur_i, blur_f = 3.0, 2.5
supp_update = 20

niter_store_errors = 10  # number of real/fourier error metric points

#recon_intens = frame.copy()
recon_intens = np.load('lin_comb_fit_4.npy')
recon_mask = mask_emc.copy()

#gt_intens = np.load('intensity.npy')[:]
#gt_mask = np.ones_like(recon_intens)
#recon_intens = gt_intens
#recon_mask = gt_mask

# Initialisation of arrays
def_rng = np.random.default_rng()

recon_real_array = np.zeros(shape=(n_recons, niter_store, *recon_intens.shape))
recon_phase_array = np.zeros(shape=(n_recons, niter_store, *recon_intens.shape))

recon_real_array_nosupp = np.zeros(shape=(n_recons, niter_store, *recon_intens.shape))
recon_phase_array_nosupp = np.zeros(shape=(n_recons, niter_store, *recon_intens.shape))

recon_fourier_array = np.zeros(shape=(n_recons, niter_store, *recon_intens.shape))
recon_fourier_poiss_array = np.zeros(shape=(n_recons, niter_store, *recon_intens.shape))

support_array = np.zeros(shape=(n_recons, niter_store, *recon_intens.shape))

error_real_array = np.zeros(shape=(n_recons, niter_store_errors))
error_fourier_array = np.zeros(shape=(n_recons, niter_store_errors))

#constraints_list = ['enforce_positivity', 'enforce_real']
#constraints_list = ['enforce_real']
#constraints_list = ['enforce_positivity']
constraints_list = []

# Phasing loop
time_now = time.localtime(time.time())
for i in range(n_recons):
    write_text(f"\rRunning reconstruction {i+1}/{n_recons}...")
    phaser = spimage.Reconstructor()

    # Initialising iterations and output parameters
    phaser.set_number_of_iterations(niter_alg + niter_er)
    phaser.set_number_of_outputs_images(niter_store)
    phaser.set_number_of_outputs_scores(niter_store_errors)

    # Initial support
    phaser.set_initial_support(support_mask=support_cyl)

    # Fourier space mask - only for EMC reconstructions
    phaser.set_mask(recon_mask)

    # Setting intensity to be phased
    phaser.set_intensities(recon_intens)

    # Support algorithm
    phaser.append_support_algorithm(
        "area",
        blur_init=blur_i,
        blur_final=blur_f,
        area_init=volume_i,
        area_final=volume_f,
        update_period=supp_update,
        number_of_iterations=niter_alg,
    )

    phaser.append_support_algorithm(
        "area",
        blur_init=blur_f,
        blur_final=blur_f,
        area_init=volume_f,
        area_final=volume_f,
        update_period=100,
        number_of_iterations=niter_er,
    )

    # Phasing algorithms
    if alg == "diffmap":
        phaser.append_phasing_algorithm(
            alg,
            constraints=constraints_list,
            beta_init=beta_start,
            beta_final=beta_end,
            number_of_iterations=niter_alg,
            gamma1=-1 / beta_start,
            gamma2=(3 - beta_start) / (2 * beta_start),
        )
    else:
        phaser.append_phasing_algorithm(
            alg,
            constraints=constraints_list,
            beta_init=beta_start,
            beta_final=beta_end,
            number_of_iterations=niter_alg,
        )

    phaser.append_phasing_algorithm(
        "er", constraints=constraints_list, number_of_iterations=niter_er
    )

    output = phaser.reconstruct()

    # Retrieving phasing results
    recon_real = output["real_space"]
    recon_fourier = output["fourier_space"]
    support = output["support"]
    error_real = output["real_error"]
    error_fourier = output["fourier_error"]

    # Enforcing support constraint for density
    object_density = np.abs(recon_real)
    object_density[support == False] = 0.0

    object_phase = np.angle(recon_real)
    object_phase[support == False] = 0.0

    # Not enforcing support constraint for density
    object_density_nsp = np.abs(recon_real)
    object_phase_nsp = np.angle(recon_real)

    object_fourier = np.abs(recon_fourier) ** 2
    object_fourier_poiss = def_rng.poisson(lam=object_fourier)

    # Storing reconstructions results in arrays
    recon_real_array[i] = object_density
    recon_phase_array[i] = object_phase
    recon_real_array_nosupp[i] = object_density_nsp
    recon_phase_array_nosupp[i] = object_phase_nsp

    recon_fourier_array[i] = object_fourier
    recon_fourier_poiss_array[i] = object_fourier_poiss
    support_array[i] = support
    error_real_array[i] = error_real
    error_fourier_array[i] = error_fourier

    clear_output(wait=True)

write_text(f"\rAll {n_recons} reconstruction(s) finished!!!")